# Gold tool paths

The gold tool path for each of the 50 questions, run through `tools.py` with every tool
output printed. Each path is written per question, not per template: two questions from
the same template take different steps whenever the question or the data asks for
something different (a positive rate instead of a negative one, an absent segment that
has to be shown as empty, a joint top issue that has to be tested twice).

Every path follows the method already verified in `gold_answers_manual.ipynb`, and stops
where the agent would start reasoning: the printed tables carry every figure the gold
answer quotes. Arithmetic *between* tool outputs (a gap in percentage points, a rate
minus the industry average) is the agent's job and is labelled where it appears.

The first reasoning step - reading the industry, organisation and topic out of the
question - is hardcoded in each cell, so what is being checked here is the analysis, not
the parameter extraction.

Expects in the working directory: `tools.py`, `question_set.csv`, `easy_data.csv`,
`hard_data.csv` and `gold_answers.csv`.


In [1]:
import textwrap

import pandas as pd
from tools import Tools, MIN_N, NON_ACTIONABLE

pd.set_option("display.width", 300)
pd.set_option("display.max_columns", 25)

questions = pd.read_csv("question_set.csv", encoding="utf-8-sig")
DATA = {"easy": pd.read_csv("easy_data.csv"), "hard": pd.read_csv("hard_data.csv")}

# the answers these paths have to support, one per task
GOLD_ANSWERS = (pd.read_csv("gold_answers.csv", encoding="utf-8-sig")
                  .set_index("task_id")["gold_answer"])

print(f"{len(questions)} questions | easy {len(DATA['easy'])} rows | hard {len(DATA['hard'])} rows")
questions.head()

50 questions | easy 25650 rows | hard 13216 rows


,task_id,dataset,question
0,T1-E1,easy,What proportion of Banking reviews that mentio...
1,T1-E2,easy,What proportion of Fashion reviews that mentio...
2,T1-H1,hard,What proportion of Fashion reviews that mentio...
3,T1-H2,hard,What proportion of Price Comparison reviews th...
4,T1-H3,hard,What proportion of Ride Hailing reviews that m...


## Helper functions

`start` opens a fresh workspace on the dataset the question is asked of, `report` prints every tool call and its output in order, and `gaps` does the one piece of arithmetic T9 needs between two tool outputs.


In [2]:
RUNS = {}   # task_id -> the Tools instance that answered it, for the path summary below


def start(task_id):
    """Print the question and open a fresh tool workspace on the dataset it is asked of."""
    q = questions.loc[questions.task_id == task_id].iloc[0]
    print("=" * 100)
    print(f"{task_id}  |  dataset: {q.dataset}")
    print(textwrap.fill(q.question, width=100))
    print("=" * 100)
    tools = Tools(DATA[q.dataset])
    RUNS[task_id] = tools
    return tools


# Arguments a tool fills in for itself. Dropping them leaves the call an agent would
# actually have to make, rather than the tool's full parameter list.
DEFAULT_ARGS = {("summarise", "min_n_column"): "total_count",
                ("rank", "ascending"): False,
                ("ztest", "alternative"): "two-sided"}


def signature(call):
    """A logged call written back out as it was made: filter(industry='Banking', ...).

    Tools does not log arguments passed as None, so a rank call with no top_k in its log
    is one that asked for the whole ranking; it is restored here, since a path replayed
    without it would silently come back cut to the top 3.
    """
    tool, args = call["tool"], call["args"]
    shown = {k: v for k, v in args.items() if DEFAULT_ARGS.get((tool, k), object()) != v}
    if tool == "rank" and "top_k" not in args:
        shown["top_k"] = None
    return f"{tool}({', '.join(f'{k}={v!r}' for k, v in shown.items())})"


def report(tools):
    """Print every tool call made so far and its output, in order."""
    for i, call in enumerate(tools.calls, 1):
        print(f"\n[{i}] {signature(call)}")
        result = call["result"]
        if "error" in result:
            print("     ERROR:", result["error"])
            continue
        if "rows" in result:
            frame = pd.DataFrame(result["rows"])
            print(frame.to_string(index=False) if len(frame) else "     (no rows)")
            for key in ("selection_rows", "below_min_n", "excluded", "note", "n_groups_available"):
                if key in result:
                    print(f"     {key}: {result[key]}")
        else:
            print("    ", {k: v for k, v in result.items() if k != "id"})
    print(f"\ntool path: {' > '.join(tools.path())}")


def gaps(tools, ref_a, ref_b):
    """Agent arithmetic, not a tool call: for each topic in A's top 3, A's negative rate
    minus B's rate on the same topic. B has no rate where the topic missed the floor."""
    a, b = tools.store[ref_a], tools.store[ref_b]
    rate_b = dict(zip(b["group"], b["neg_rate"]))
    rows = [{"topic": g, "neg_rate_a": round(ra, 4),
             "neg_rate_b": round(rate_b[g], 4) if g in rate_b else None,
             "gap_pp": round((ra - rate_b[g]) * 100, 1) if g in rate_b else None,
             "actionable": g not in NON_ACTIONABLE}
            for g, ra in zip(a["group"], a["neg_rate"])]
    out = pd.DataFrame(rows).sort_values("gap_pp", ascending=False, na_position="last")
    print("\nAGENT ARITHMETIC: A's top 3 topics vs B on the same topic")
    print(out.to_string(index=False))

## Answer all 50 questions

## T1 - Single value retrieval

In [3]:
# ANSWER NEEDS: negative rate of the slice, and the counts behind it
t = start("T1-E1")

t.filter(industry="Banking", aspect="app-website")            # -> s1
t.summarise("s1")                                             # neg_rate 80/427

report(t)

T1-E1  |  dataset: easy
What proportion of Banking reviews that mention the app or website are negative?

[1] filter(industry='Banking', aspect='app-website')
     {'n_rows': 427}

[2] summarise(ref='s1')
group  total_count  neg_count  neu_count  pos_count  neg_rate  neu_rate  pos_rate  prevalence  priority_score
  all          427         80        110        237    0.1874    0.2576     0.555         1.0          0.1874
     selection_rows: 427

tool path: filter > summarise


In [4]:
# ANSWER NEEDS: POSITIVE rate this time, not the negative one
t = start("T1-E2")

t.filter(industry="Fashion", aspect="ease-of-use")            # -> s1
t.summarise("s1")                                             # read pos_rate 402/463

report(t)

T1-E2  |  dataset: easy
What proportion of Fashion reviews that mention ease of use are positive?

[1] filter(industry='Fashion', aspect='ease-of-use')
     {'n_rows': 463}

[2] summarise(ref='s1')
group  total_count  neg_count  neu_count  pos_count  neg_rate  neu_rate  pos_rate  prevalence  priority_score
  all          463         60          1        402    0.1296    0.0022    0.8683         1.0          0.1296
     selection_rows: 463

tool path: filter > summarise


In [5]:
# ANSWER NEEDS: negative rate of the slice, and the counts behind it
t = start("T1-H1")

t.filter(industry="Fashion", aspect="speed")                  # -> s1
t.summarise("s1")                                             # neg_rate 117/549

report(t)

T1-H1  |  dataset: hard
What proportion of Fashion reviews that mention speed are negative?

[1] filter(industry='Fashion', aspect='speed')
     {'n_rows': 549}

[2] summarise(ref='s1')
group  total_count  neg_count  neu_count  pos_count  neg_rate  neu_rate  pos_rate  prevalence  priority_score
  all          549        117          1        431    0.2131    0.0018    0.7851         1.0          0.2131
     selection_rows: 549

tool path: filter > summarise


In [6]:
# ANSWER NEEDS: negative rate of the slice, and the counts behind it
t = start("T1-H2")

t.filter(industry="Price Comparison", aspect="attitude-of-staff")   # -> s1
t.summarise("s1")                                             # neg_rate 43/396

report(t)

T1-H2  |  dataset: hard
What proportion of Price Comparison reviews that mention staff attitude are negative?

[1] filter(industry='Price Comparison', aspect='attitude-of-staff')
     {'n_rows': 396}

[2] summarise(ref='s1')
group  total_count  neg_count  neu_count  pos_count  neg_rate  neu_rate  pos_rate  prevalence  priority_score
  all          396         43          2        351    0.1086    0.0051    0.8864         1.0          0.1086
     selection_rows: 396

tool path: filter > summarise


In [7]:
# ANSWER NEEDS: the slice size, which is what makes this unanswerable (3 < 30 reviews)
t = start("T1-H3")

t.filter(industry="Ride Hailing", aspect="discounts-promotions")    # -> s1, only 3 rows
t.summarise("s1")                                             # a rate exists but is not reportable

report(t)

T1-H3  |  dataset: hard
What proportion of Ride Hailing reviews that mention discounts and promotions are negative?

[1] filter(industry='Ride Hailing', aspect='discounts-promotions')
     {'n_rows': 3}

[2] summarise(ref='s1')
group  total_count  neg_count  neu_count  pos_count  neg_rate  neu_rate  pos_rate  prevalence  priority_score
  all            3          2          0          1    0.6667       0.0    0.3333         1.0          0.6667
     selection_rows: 3

tool path: filter > summarise


## T2 - Distribution

In [8]:
# ANSWER NEEDS: positive, negative AND neutral rates, plus the number of mentions
t = start("T2-E1")

t.filter(industry="Ride Hailing")                       # -> s1
t.summarise("s1")                      # pos_rate / neg_rate / neu_rate + total_count

report(t)

T2-E1  |  dataset: easy
What is the sentiment breakdown across all Ride Hailing reviews?

[1] filter(industry='Ride Hailing')
     {'n_rows': 1532}

[2] summarise(ref='s1')
group  total_count  neg_count  neu_count  pos_count  neg_rate  neu_rate  pos_rate  prevalence  priority_score
  all         1532       1115         79        338    0.7278    0.0516    0.2206         1.0          0.7278
     selection_rows: 1532

tool path: filter > summarise


In [9]:
# ANSWER NEEDS: positive, negative AND neutral rates, plus the number of mentions
t = start("T2-E2")

t.filter(org="Marbrook")                       # -> s1
t.summarise("s1")                      # pos_rate / neg_rate / neu_rate + total_count

report(t)

T2-E2  |  dataset: easy
What is the sentiment breakdown across all Marbrook reviews?

[1] filter(org='Marbrook')
     {'n_rows': 1044}

[2] summarise(ref='s1')
group  total_count  neg_count  neu_count  pos_count  neg_rate  neu_rate  pos_rate  prevalence  priority_score
  all         1044        342         17        685    0.3276    0.0163    0.6561         1.0          0.3276
     selection_rows: 1044

tool path: filter > summarise


In [10]:
# ANSWER NEEDS: positive, negative AND neutral rates, plus the number of mentions
t = start("T2-H1")

t.filter(org="Vanter Financial")                       # -> s1
t.summarise("s1")                      # pos_rate / neg_rate / neu_rate + total_count

report(t)

T2-H1  |  dataset: hard
What is the sentiment breakdown across all Vanter Financial reviews?

[1] filter(org='Vanter Financial')
     {'n_rows': 420}

[2] summarise(ref='s1')
group  total_count  neg_count  neu_count  pos_count  neg_rate  neu_rate  pos_rate  prevalence  priority_score
  all          420        119         48        253    0.2833    0.1143    0.6024         1.0          0.2833
     selection_rows: 420

tool path: filter > summarise


In [11]:
# ANSWER NEEDS: positive, negative AND neutral rates, plus the number of mentions
t = start("T2-H2")

t.filter(org="PricePilot")                       # -> s1
t.summarise("s1")                      # pos_rate / neg_rate / neu_rate + total_count

report(t)

T2-H2  |  dataset: hard
What is the sentiment breakdown across all PricePilot reviews?

[1] filter(org='PricePilot')
     {'n_rows': 805}

[2] summarise(ref='s1')
group  total_count  neg_count  neu_count  pos_count  neg_rate  neu_rate  pos_rate  prevalence  priority_score
  all          805        125          3        677    0.1553    0.0037     0.841         1.0          0.1553
     selection_rows: 805

tool path: filter > summarise


In [12]:
# ANSWER NEEDS: positive, negative AND neutral rates, plus the number of mentions
t = start("T2-H3")

t.filter(org="Sable Row")                       # -> s1
t.summarise("s1")                      # pos_rate / neg_rate / neu_rate + total_count

report(t)

T2-H3  |  dataset: hard
What is the sentiment breakdown across all Sable Row reviews?

[1] filter(org='Sable Row')
     {'n_rows': 925}

[2] summarise(ref='s1')
group  total_count  neg_count  neu_count  pos_count  neg_rate  neu_rate  pos_rate  prevalence  priority_score
  all          925        279          3        643    0.3016    0.0032    0.6951         1.0          0.3016
     selection_rows: 925

tool path: filter > summarise


## T3 - Ranking by volume

In [13]:
# ANSWER NEEDS: the 3 topics with the most complaints, and each complaint count
t = start("T3-E1")

t.filter(industry="Travel Booking", sentiment="negative")   # -> s1, complaints only
t.summarise("s1", group_by="aspect")                # -> t1, total_count = complaints
t.rank("t1", by="total_count", top_k=3)             # -> t2, ranked by volume

report(t)

T3-E1  |  dataset: easy
What are the top 3 topics with the most number of complaints in Travel Booking?

[1] filter(industry='Travel Booking', sentiment='negative')
     {'n_rows': 1145}

[2] summarise(ref='s1', group_by='aspect')
                group  total_count  neg_count  neu_count  pos_count  neg_rate  neu_rate  pos_rate  prevalence  priority_score
          app-website          215        215          0          0       1.0       0.0       0.0      0.1878          0.1878
       account-access          149        149          0          0       1.0       0.0       0.0      0.1301          0.1301
                email          138        138          0          0       1.0       0.0       0.0      0.1205          0.1205
                phone          116        116          0          0       1.0       0.0       0.0      0.1013          0.1013
 general-satisfaction          103        103          0          0       1.0       0.0       0.0      0.0900          0.0900
    attitude-

In [14]:
# ANSWER NEEDS: the 3 topics with the most complaints, and each complaint count
t = start("T3-E2")

t.filter(industry="Groceries", sentiment="negative")   # -> s1, complaints only
t.summarise("s1", group_by="aspect")                # -> t1, total_count = complaints
t.rank("t1", by="total_count", top_k=3)             # -> t2, ranked by volume

report(t)

T3-E2  |  dataset: easy
What are the top 3 topics with the most number of complaints in Groceries?

[1] filter(industry='Groceries', sentiment='negative')
     {'n_rows': 1286}

[2] summarise(ref='s1', group_by='aspect')
                group  total_count  neg_count  neu_count  pos_count  neg_rate  neu_rate  pos_rate  prevalence  priority_score
          app-website          383        383          0          0       1.0       0.0       0.0      0.2978          0.2978
          ease-of-use          215        215          0          0       1.0       0.0       0.0      0.1672          0.1672
                email          100        100          0          0       1.0       0.0       0.0      0.0778          0.0778
              reviews           98         98          0          0       1.0       0.0       0.0      0.0762          0.0762
                phone           96         96          0          0       1.0       0.0       0.0      0.0747          0.0747
 general-satisfaction  

In [15]:
# ANSWER NEEDS: the 3 topics with the most complaints, and each complaint count
t = start("T3-H1")

t.filter(industry="Fashion", sentiment="negative")   # -> s1, complaints only
t.summarise("s1", group_by="aspect")                # -> t1, total_count = complaints
t.rank("t1", by="total_count", top_k=3)             # -> t2, ranked by volume

report(t)

T3-H1  |  dataset: hard
What are the top 3 topics with the most number of complaints in Fashion?

[1] filter(industry='Fashion', sentiment='negative')
     {'n_rows': 894}

[2] summarise(ref='s1', group_by='aspect')
                group  total_count  neg_count  neu_count  pos_count  neg_rate  neu_rate  pos_rate  prevalence  priority_score
          app-website          241        241          0          0       1.0       0.0       0.0      0.2696          0.2696
 general-satisfaction          160        160          0          0       1.0       0.0       0.0      0.1790          0.1790
                speed          117        117          0          0       1.0       0.0       0.0      0.1309          0.1309
    attitude-of-staff          116        116          0          0       1.0       0.0       0.0      0.1298          0.1298
                phone           70         70          0          0       1.0       0.0       0.0      0.0783          0.0783
price-value-for-money       

In [16]:
# ANSWER NEEDS: the 3 topics with the most complaints, and each complaint count
t = start("T3-H2")

t.filter(industry="Price Comparison", sentiment="negative")   # -> s1, complaints only
t.summarise("s1", group_by="aspect")                # -> t1, total_count = complaints
t.rank("t1", by="total_count", top_k=3)             # -> t2, ranked by volume

report(t)

T3-H2  |  dataset: hard
What are the top 3 topics with the most number of complaints in Price Comparison?

[1] filter(industry='Price Comparison', sentiment='negative')
     {'n_rows': 509}

[2] summarise(ref='s1', group_by='aspect')
                group  total_count  neg_count  neu_count  pos_count  neg_rate  neu_rate  pos_rate  prevalence  priority_score
          app-website          148        148          0          0       1.0       0.0       0.0      0.2908          0.2908
price-value-for-money           65         65          0          0       1.0       0.0       0.0      0.1277          0.1277
 general-satisfaction           48         48          0          0       1.0       0.0       0.0      0.0943          0.0943
                phone           45         45          0          0       1.0       0.0       0.0      0.0884          0.0884
    attitude-of-staff           43         43          0          0       1.0       0.0       0.0      0.0845          0.0845
 discounts

In [17]:
# ANSWER NEEDS: how many topics exist at all - a top 3 needs 3
t = start("T3-H3")

t.filter(industry="Consulting", sentiment="negative")   # -> s1
t.summarise("s1", group_by="aspect")                    # -> t1, one topic only
t.rank("t1", by="total_count", top_k=3)                 # -> t2, note: fewer than 3 available

report(t)

T3-H3  |  dataset: hard
What are the top 3 topics with the most number of complaints in Consulting?

[1] filter(industry='Consulting', sentiment='negative')
     {'n_rows': 52}

[2] summarise(ref='s1', group_by='aspect')
         group  total_count  neg_count  neu_count  pos_count  neg_rate  neu_rate  pos_rate  prevalence  priority_score
account-access           52         52          0          0       1.0       0.0       0.0         1.0             1.0
     selection_rows: 52

[3] rank(ref='t1', by='total_count', top_k=3)
         group  total_count  neg_count  neu_count  pos_count  neg_rate  neu_rate  pos_rate  prevalence  priority_score  rank
account-access           52         52          0          0       1.0       0.0       0.0         1.0             1.0     1
     n_groups_available: 1

tool path: filter > summarise > rank


## T4 - Cross-segment comparison

In [18]:
# ANSWER NEEDS: both industries' negative rates on the topic, side by side
t = start("T4-E1")

t.filter(aspect="price-value-for-money", industry=['Banking', 'Ride Hailing'])   # -> s1
t.summarise("s1", group_by="industry")         # -> t1, one row per industry

report(t)

T4-E1  |  dataset: easy
Looking only at reviews about price and value for money, is the negative sentiment rate higher in
Banking or Ride Hailing?

[1] filter(industry=['Banking', 'Ride Hailing'], aspect='price-value-for-money')
     {'n_rows': 280}

[2] summarise(ref='s1', group_by='industry')
       group  total_count  neg_count  neu_count  pos_count  neg_rate  neu_rate  pos_rate  prevalence  priority_score
     Banking          160         32          0        128    0.2000    0.0000     0.800      0.5714          0.1143
Ride Hailing          120         91          8         21    0.7583    0.0667     0.175      0.4286          0.3250
     selection_rows: 280

tool path: filter > summarise


In [19]:
# ANSWER NEEDS: both industries' negative rates on the topic, side by side
t = start("T4-E2")

t.filter(aspect="app-website", industry=['Fashion', 'Travel Booking'])   # -> s1
t.summarise("s1", group_by="industry")         # -> t1, one row per industry

report(t)

T4-E2  |  dataset: easy
Looking only at reviews about the app or website, is the negative sentiment rate higher in Fashion
or Travel Booking?

[1] filter(industry=['Fashion', 'Travel Booking'], aspect='app-website')
     {'n_rows': 1276}

[2] summarise(ref='s1', group_by='industry')
         group  total_count  neg_count  neu_count  pos_count  neg_rate  neu_rate  pos_rate  prevalence  priority_score
       Fashion          788        220          4        564    0.2792    0.0051    0.7157      0.6176          0.1724
Travel Booking          488        215          4        269    0.4406    0.0082    0.5512      0.3824          0.1685
     selection_rows: 1276

tool path: filter > summarise


In [20]:
# ANSWER NEEDS: both industries' negative rates on the topic, side by side
t = start("T4-H1")

t.filter(aspect="ease-of-use", industry=['Groceries', 'Trading'])   # -> s1
t.summarise("s1", group_by="industry")         # -> t1, one row per industry

report(t)

T4-H1  |  dataset: hard
Looking only at reviews about ease of use, is the negative sentiment rate higher in Groceries or
Trading?

[1] filter(industry=['Groceries', 'Trading'], aspect='ease-of-use')
     {'n_rows': 790}

[2] summarise(ref='s1', group_by='industry')
    group  total_count  neg_count  neu_count  pos_count  neg_rate  neu_rate  pos_rate  prevalence  priority_score
Groceries          523        222          1        300    0.4245    0.0019    0.5736       0.662          0.2810
  Trading          267         29          3        235    0.1086    0.0112    0.8801       0.338          0.0367
     selection_rows: 790

tool path: filter > summarise


In [21]:
# ANSWER NEEDS: Banking's rate, and proof that Consulting has nothing to compare it with
t = start("T4-H2")

# the two sides are filtered separately: a grouped summary would simply omit the
# empty industry, and "no row" is easy to misread as "no complaints"
t.filter(aspect="ease-of-use", industry="Banking")      # -> s1, 383 rows
t.summarise("s1")                                       # -> t1, Banking 25.1% negative
t.filter(aspect="ease-of-use", industry="Consulting")   # -> s2, 0 rows: nothing to compare

report(t)

T4-H2  |  dataset: hard
Looking only at reviews about ease of use, is the negative sentiment rate higher in Banking or
Consulting?

[1] filter(industry='Banking', aspect='ease-of-use')
     {'n_rows': 383}

[2] summarise(ref='s1')
group  total_count  neg_count  neu_count  pos_count  neg_rate  neu_rate  pos_rate  prevalence  priority_score
  all          383         96          0        287    0.2507       0.0    0.7493         1.0          0.2507
     selection_rows: 383

[3] filter(industry='Consulting', aspect='ease-of-use')
     {'n_rows': 0}

tool path: filter > summarise > filter


In [22]:
# ANSWER NEEDS: proof that neither industry has any reviews on the topic
t = start("T4-H3")

t.filter(aspect="price-value-for-money", industry="Consulting")   # -> s1, 0 rows
t.filter(aspect="price-value-for-money", industry="Streaming")    # -> s2, 0 rows

report(t)

T4-H3  |  dataset: hard
Looking only at reviews about price and value for money, is the negative sentiment rate higher in
Consulting or Streaming?

[1] filter(industry='Consulting', aspect='price-value-for-money')
     {'n_rows': 0}

[2] filter(industry='Streaming', aspect='price-value-for-money')
     {'n_rows': 0}

tool path: filter > filter


## T5 - Statistical testing

In [23]:
# ANSWER NEEDS: both rates, z, p, and whether the four cells are large enough to trust the test
t = start("T5-E1")

t.filter(org="CompareHive", aspect="attitude-of-staff")   # -> s1
t.filter(org="Tallywise", aspect="attitude-of-staff")   # -> s2
t.ztest("s1", "s2")                        # rates, z, p, sufficient_data

report(t)

T5-E1  |  dataset: easy
Do CompareHive and Tallywise have significantly different negative sentiment rates for staff
attitude?

[1] filter(org='CompareHive', aspect='attitude-of-staff')
     {'n_rows': 103}

[2] filter(org='Tallywise', aspect='attitude-of-staff')
     {'n_rows': 118}

[3] ztest(ref_a='s1', ref_b='s2')
     {'group_a': {'total': 103, 'neg': 11, 'non_neg': 92, 'neg_rate': 0.1068}, 'group_b': {'total': 118, 'neg': 18, 'non_neg': 100, 'neg_rate': 0.1525}, 'gap_pp': -4.6, 'z': -1.005, 'p_value': 0.315, 'significant': False}

tool path: filter > filter > ztest


In [24]:
# ANSWER NEEDS: both rates, z, p, and whether the four cells are large enough to trust the test
t = start("T5-E2")

t.filter(org="Trippa", aspect="general-satisfaction")   # -> s1
t.filter(org="Roamly", aspect="general-satisfaction")   # -> s2
t.ztest("s1", "s2")                        # rates, z, p, sufficient_data

report(t)

T5-E2  |  dataset: easy
Do Trippa and Roamly have significantly different negative sentiment rates for general satisfaction?

[1] filter(org='Trippa', aspect='general-satisfaction')
     {'n_rows': 123}

[2] filter(org='Roamly', aspect='general-satisfaction')
     {'n_rows': 95}

[3] ztest(ref_a='s1', ref_b='s2')
     {'group_a': {'total': 123, 'neg': 54, 'non_neg': 69, 'neg_rate': 0.439}, 'group_b': {'total': 95, 'neg': 21, 'non_neg': 74, 'neg_rate': 0.2211}, 'gap_pp': 21.8, 'z': 3.359, 'p_value': 0.0008, 'significant': True}

tool path: filter > filter > ztest


In [25]:
# ANSWER NEEDS: both rates, z, p, and whether the four cells are large enough to trust the test
t = start("T5-H1")

t.filter(industry="Groceries", aspect="app-website")   # -> s1
t.filter(industry="Price Comparison", aspect="app-website")   # -> s2
t.ztest("s1", "s2")                        # rates, z, p, sufficient_data

report(t)

T5-H1  |  dataset: hard
Do Groceries and Price Comparison have significantly different negative sentiment rates for the app
or website?

[1] filter(industry='Groceries', aspect='app-website')
     {'n_rows': 451}

[2] filter(industry='Price Comparison', aspect='app-website')
     {'n_rows': 312}

[3] ztest(ref_a='s1', ref_b='s2')
     {'group_a': {'total': 451, 'neg': 217, 'non_neg': 234, 'neg_rate': 0.4812}, 'group_b': {'total': 312, 'neg': 148, 'non_neg': 164, 'neg_rate': 0.4744}, 'gap_pp': 0.7, 'z': 0.185, 'p_value': 0.8535, 'significant': False}

tool path: filter > filter > ztest


In [26]:
# ANSWER NEEDS: the four cells - they are what makes the test unusable here
t = start("T5-H2")

t.filter(org="Halden Savings", aspect="price-value-for-money")   # -> s1
t.filter(org="Kestrel Bank", aspect="price-value-for-money")   # -> s2
t.ztest("s1", "s2")                        # rates, z, p, sufficient_data

report(t)

T5-H2  |  dataset: hard
Do Halden Savings and Kestrel Bank have significantly different negative sentiment rates for price
and value for money?

[1] filter(org='Halden Savings', aspect='price-value-for-money')
     {'n_rows': 29}

[2] filter(org='Kestrel Bank', aspect='price-value-for-money')
     {'n_rows': 20}

[3] ztest(ref_a='s1', ref_b='s2')
     {'group_a': {'total': 29, 'neg': 16, 'non_neg': 13, 'neg_rate': 0.5517}, 'group_b': {'total': 20, 'neg': 6, 'non_neg': 14, 'neg_rate': 0.3}, 'gap_pp': 25.2, 'z': 1.741, 'p_value': 0.0817, 'significant': False}

tool path: filter > filter > ztest


In [27]:
# ANSWER NEEDS: the four cells - they are what makes the test unusable here
t = start("T5-H3")

t.filter(org="Northpeak Trading", aspect="speed")   # -> s1
t.filter(org="Quantly", aspect="speed")   # -> s2
t.ztest("s1", "s2")                        # rates, z, p, sufficient_data

report(t)

T5-H3  |  dataset: hard
Do Northpeak Trading and Quantly have significantly different negative sentiment rates for speed?



[1] filter(org='Northpeak Trading', aspect='speed')
     {'n_rows': 20}

[2] filter(org='Quantly', aspect='speed')
     {'n_rows': 25}

[3] ztest(ref_a='s1', ref_b='s2')
     {'group_a': {'total': 20, 'neg': 12, 'non_neg': 8, 'neg_rate': 0.6}, 'group_b': {'total': 25, 'neg': 13, 'non_neg': 12, 'neg_rate': 0.52}, 'gap_pp': 8.0, 'z': 0.537, 'p_value': 0.5915, 'significant': False}

tool path: filter > filter > ztest


## T6 - Organisation vs industry comparison

Note:
- In v1 of the tool set, the agent is expected to perform the subtraction of the negative rate from the industry average rate without the use of any tools. If agents fail at this, then we can add another tool for comparison.

In [28]:
# ANSWER NEEDS: the industry average, and every qualifying organisation's rate to compare with it
t = start("T6-E1")

t.filter(industry="Banking", aspect="app-website")        # -> s1
t.summarise("s1")                                  # -> t1, the industry average
t.summarise("s1", group_by="org", min_n=MIN_N)     # -> t2, orgs with >= 30 mentions
t.rank("t2", by="neg_rate", top_k=None)            # -> t3, worst first
# each organisation's distance from the industry average is the subtraction
# between t1 and t3 - agent arithmetic, not a tool call

report(t)

T6-E1  |  dataset: easy
How do individual organisations in Banking compare to the industry average on the app or website?
Give me each organisation's negative sentiment rate and ignore any organisations with less than 30
mentions about the app or website.

[1] filter(industry='Banking', aspect='app-website')
     {'n_rows': 427}

[2] summarise(ref='s1')
group  total_count  neg_count  neu_count  pos_count  neg_rate  neu_rate  pos_rate  prevalence  priority_score
  all          427         80        110        237    0.1874    0.2576     0.555         1.0          0.1874
     selection_rows: 427

[3] summarise(ref='s1', group_by='org', min_n=30)
           group  total_count  neg_count  neu_count  pos_count  neg_rate  neu_rate  pos_rate  prevalence  priority_score
  Northeast Bank          140         28         34         78    0.2000    0.2429    0.5571      0.3279          0.0656
    Kestrel Bank          139         17         43         79    0.1223    0.3094    0.5683      0.3255  

In [29]:
# ANSWER NEEDS: the industry average, and every qualifying organisation's rate to compare with it
t = start("T6-E2")

t.filter(industry="Travel Booking", aspect="general-satisfaction")        # -> s1
t.summarise("s1")                                  # -> t1, the industry average
t.summarise("s1", group_by="org", min_n=MIN_N)     # -> t2, orgs with >= 30 mentions
t.rank("t2", by="neg_rate", top_k=None)            # -> t3, worst first
# each organisation's distance from the industry average is the subtraction
# between t1 and t3 - agent arithmetic, not a tool call

report(t)

T6-E2  |  dataset: easy
How do individual organisations in Travel Booking compare to the industry average on general
satisfaction? Give me each organisation's negative sentiment rate and ignore any organisations with
less than 30 mentions about general satisfaction.

[1] filter(industry='Travel Booking', aspect='general-satisfaction')
     {'n_rows': 404}

[2] summarise(ref='s1')
group  total_count  neg_count  neu_count  pos_count  neg_rate  neu_rate  pos_rate  prevalence  priority_score
  all          404        103          2        299     0.255     0.005    0.7401         1.0           0.255
     selection_rows: 404

[3] summarise(ref='s1', group_by='org', min_n=30)
  group  total_count  neg_count  neu_count  pos_count  neg_rate  neu_rate  pos_rate  prevalence  priority_score
 Trippa          123         54          0         69    0.4390    0.0000    0.5610      0.3045          0.1337
Journeo          107         21          0         86    0.1963    0.0000    0.8037      0.2649  

In [30]:
# ANSWER NEEDS: the industry average, and every qualifying organisation's rate to compare with it
t = start("T6-H1")

t.filter(industry="Fashion", aspect="app-website")        # -> s1
t.summarise("s1")                                  # -> t1, the industry average
t.summarise("s1", group_by="org", min_n=MIN_N)     # -> t2, orgs with >= 30 mentions
t.rank("t2", by="neg_rate", top_k=None)            # -> t3, worst first
# each organisation's distance from the industry average is the subtraction
# between t1 and t3 - agent arithmetic, not a tool call

report(t)

T6-H1  |  dataset: hard
How do individual organisations in Fashion compare to the industry average on the app or website?
Give me each organisation's negative sentiment rate and ignore any organisations with less than 30
mentions about the app or website.

[1] filter(industry='Fashion', aspect='app-website')
     {'n_rows': 877}

[2] summarise(ref='s1')
group  total_count  neg_count  neu_count  pos_count  neg_rate  neu_rate  pos_rate  prevalence  priority_score
  all          877        241          4        632    0.2748    0.0046    0.7206         1.0          0.2748
     selection_rows: 877

[3] summarise(ref='s1', group_by='org', min_n=30)
     group  total_count  neg_count  neu_count  pos_count  neg_rate  neu_rate  pos_rate  prevalence  priority_score
  Marbrook          276        103          0        173    0.3732    0.0000    0.6268      0.3147          0.1174
 Sable Row          253         46          2        205    0.1818    0.0079    0.8103      0.2885          0.0525
 No

In [31]:
# ANSWER NEEDS: proof that the industry has no reviews on this topic at all
t = start("T6-H2")

t.filter(industry="Consulting", aspect="app-website")   # -> s1, 0 rows: no org to compare

report(t)

T6-H2  |  dataset: hard
How do individual organisations in Consulting compare to the industry average on the app or website?
Give me each organisation's negative sentiment rate and ignore any organisations with less than 30
mentions about the app or website.

[1] filter(industry='Consulting', aspect='app-website')
     {'n_rows': 0}

tool path: filter


In [32]:
# ANSWER NEEDS: proof that the industry has no reviews on this topic at all
t = start("T6-H3")

t.filter(industry="Streaming", aspect="general-satisfaction")   # -> s1, 0 rows

report(t)

T6-H3  |  dataset: hard
How do individual organisations in Streaming compare to the industry average on general
satisfaction? Give me each organisation's negative sentiment rate and ignore any organisations with
less than 30 mentions about general satisfaction.

[1] filter(industry='Streaming', aspect='general-satisfaction')
     {'n_rows': 0}

tool path: filter


## T7 - Driver identification by severity

In [33]:
# ANSWER NEEDS: every qualifying topic ranked by negative rate, worst first
t = start("T7-E1")

t.filter(org="Northeast Bank")                              # -> s1
t.summarise("s1", group_by="aspect", min_n=MIN_N)   # -> t1, the 30-mention floor
t.rank("t1", by="neg_rate", top_k=None)             # -> t2, the full ranking

report(t)

T7-E1  |  dataset: easy
What is the biggest driver of dissatisfaction at Northeast Bank? Rank topics by negative sentiment
rate and ignore any topics with less than 30 mentions.

[1] filter(org='Northeast Bank')
     {'n_rows': 642}

[2] summarise(ref='s1', group_by='aspect', min_n=30)
                group  total_count  neg_count  neu_count  pos_count  neg_rate  neu_rate  pos_rate  prevalence  priority_score
          app-website          140         28         34         78    0.2000    0.2429    0.5571      0.2181          0.0436
 general-satisfaction           85          8          0         77    0.0941    0.0000    0.9059      0.1324          0.0125
          ease-of-use           57         20          0         37    0.3509    0.0000    0.6491      0.0888          0.0312
       account-access           40         25         13          2    0.6250    0.3250    0.0500      0.0623          0.0389
           competitor           40         10          0         30    0.2500    0.

In [34]:
# ANSWER NEEDS: every qualifying topic ranked by negative rate, worst first
t = start("T7-E2")

t.filter(org="Wayfare")                              # -> s1
t.summarise("s1", group_by="aspect", min_n=MIN_N)   # -> t1, the 30-mention floor
t.rank("t1", by="neg_rate", top_k=None)             # -> t2, the full ranking

report(t)

T7-E2  |  dataset: easy
What is the biggest driver of dissatisfaction at Wayfare? Rank topics by negative sentiment rate and
ignore any topics with less than 30 mentions.

[1] filter(org='Wayfare')
     {'n_rows': 644}

[2] summarise(ref='s1', group_by='aspect', min_n=30)
                group  total_count  neg_count  neu_count  pos_count  neg_rate  neu_rate  pos_rate  prevalence  priority_score
          app-website          159         82          1         76    0.5157    0.0063    0.4780      0.2469          0.1273
 general-satisfaction           79          7          0         72    0.0886    0.0000    0.9114      0.1227          0.0109
          ease-of-use           46         16          0         30    0.3478    0.0000    0.6522      0.0714          0.0248
       account-access           40         37          2          1    0.9250    0.0500    0.0250      0.0621          0.0575
           competitor           40          7          1         32    0.1750    0.0250    0.8000

In [35]:
# ANSWER NEEDS: every qualifying topic ranked by negative rate, worst first
t = start("T7-H1")

t.filter(org="Sable Row")                              # -> s1
t.summarise("s1", group_by="aspect", min_n=MIN_N)   # -> t1, the 30-mention floor
t.rank("t1", by="neg_rate", top_k=None)             # -> t2, the full ranking

report(t)

T7-H1  |  dataset: hard
What is the biggest driver of dissatisfaction at Sable Row? Rank topics by negative sentiment rate
and ignore any topics with less than 30 mentions.

[1] filter(org='Sable Row')
     {'n_rows': 925}

[2] summarise(ref='s1', group_by='aspect', min_n=30)
                group  total_count  neg_count  neu_count  pos_count  neg_rate  neu_rate  pos_rate  prevalence  priority_score
 general-satisfaction          354        109          0        245    0.3079    0.0000    0.6921      0.3827          0.1178
          app-website          253         46          2        205    0.1818    0.0079    0.8103      0.2735          0.0497
                speed          100         30          0         70    0.3000    0.0000    0.7000      0.1081          0.0324
          ease-of-use           74         21          0         53    0.2838    0.0000    0.7162      0.0800          0.0227
    attitude-of-staff           56         44          0         12    0.7857    0.0000    0.

In [36]:
# ANSWER NEEDS: every qualifying topic ranked by negative rate, worst first
t = start("T7-H2")

t.filter(org="Investa")                              # -> s1
t.summarise("s1", group_by="aspect", min_n=MIN_N)   # -> t1, the 30-mention floor
t.rank("t1", by="neg_rate", top_k=None)             # -> t2, the full ranking

report(t)

T7-H2  |  dataset: hard
What is the biggest driver of dissatisfaction at Investa? Rank topics by negative sentiment rate and
ignore any topics with less than 30 mentions.



[1] filter(org='Investa')
     {'n_rows': 566}

[2] summarise(ref='s1', group_by='aspect', min_n=30)
                group  total_count  neg_count  neu_count  pos_count  neg_rate  neu_rate  pos_rate  prevalence  priority_score
          ease-of-use          155         18          0        137    0.1161     0.000    0.8839      0.2739          0.0318
          app-website          136         21          0        115    0.1544     0.000    0.8456      0.2403          0.0371
 general-satisfaction           93         20          0         73    0.2151     0.000    0.7849      0.1643          0.0353
    attitude-of-staff           54         17          2         35    0.3148     0.037    0.6481      0.0954          0.0300
                phone           40         21          0         19    0.5250     0.000    0.4750      0.0707          0.0371
price-value-for-money           34         14          0         20    0.4118     0.000    0.5882      0.0601          0.0247
     selection_r

In [37]:
# ANSWER NEEDS: proof that no topic clears the 30-mention floor the question sets
t = start("T7-H3")

t.filter(org="Pinecast")                            # -> s1, 28 reviews in total
t.summarise("s1", group_by="aspect", min_n=MIN_N)   # -> t1, no rows survive the floor

report(t)

T7-H3  |  dataset: hard
What is the biggest driver of dissatisfaction at Pinecast? Rank topics by negative sentiment rate
and ignore any topics with less than 30 mentions.

[1] filter(org='Pinecast')
     {'n_rows': 28}

[2] summarise(ref='s1', group_by='aspect', min_n=30)
     (no rows)
     selection_rows: 28
     below_min_n: ['account-access', 'app-website', 'discounts-promotions']

tool path: filter > summarise


## T8 - Prioritisation by a named rule

Priority score is a pre-computed column in summarise. The task tests whether the agent can: 
- apply the right min volume to negative reviews (not all mentions)
- rank by priority score, not negative rate or volume 
- exclude non-actionable aspects before recommending
- return only the top 2 issues in the final answer

In [38]:
# ANSWER NEEDS: the priority score of every eligible topic
t = start("T8-E1")

t.filter(org="Wayfare")                              # -> s1
# the floor this question names is on negative reviews, not mentions
t.summarise("s1", group_by="aspect", min_n=MIN_N, min_n_column="neg_count")   # -> t1
t.rank("t1", by="priority_score", top_k=None)       # -> t2, prevalence x neg_rate

report(t)

T8-E1  |  dataset: easy
Which 2 issues should Wayfare address first? Rank aspects using this formula: priority score =
aspect prevalence x aspect negative rate. Ignore any aspects with less than 30 negative reviews.



[1] filter(org='Wayfare')
     {'n_rows': 644}

[2] summarise(ref='s1', group_by='aspect', min_n=30, min_n_column='neg_count')
         group  total_count  neg_count  neu_count  pos_count  neg_rate  neu_rate  pos_rate  prevalence  priority_score
   app-website          159         82          1         76    0.5157    0.0063     0.478      0.2469          0.1273
account-access           40         37          2          1    0.9250    0.0500     0.025      0.0621          0.0575
         email           40         34          0          6    0.8500    0.0000     0.150      0.0621          0.0528
     selection_rows: 644
     below_min_n: ['attitude-of-staff', 'competitor', 'discounts-promotions', 'ease-of-use', 'general-satisfaction', 'phone', 'price-value-for-money', 'reviews', 'speed']

[3] rank(ref='t1', by='priority_score', top_k=None)
         group  total_count  neg_count  neu_count  pos_count  neg_rate  neu_rate  pos_rate  prevalence  priority_score  rank
   app-website        

In [39]:
# ANSWER NEEDS: the priority score of every eligible topic
t = start("T8-E2")

t.filter(org="Larkmead Market")                              # -> s1
# the floor this question names is on negative reviews, not mentions
t.summarise("s1", group_by="aspect", min_n=MIN_N, min_n_column="neg_count")   # -> t1
t.rank("t1", by="priority_score", top_k=None)       # -> t2, prevalence x neg_rate

report(t)

T8-E2  |  dataset: easy
Which 2 issues should Larkmead Market address first? Rank aspects using this formula: priority score
= aspect prevalence x aspect negative rate. Ignore any aspects with less than 30 negative reviews.

[1] filter(org='Larkmead Market')
     {'n_rows': 903}

[2] summarise(ref='s1', group_by='aspect', min_n=30, min_n_column='neg_count')
      group  total_count  neg_count  neu_count  pos_count  neg_rate  neu_rate  pos_rate  prevalence  priority_score
app-website          259        124          5        130    0.4788    0.0193    0.5019      0.2868          0.1373
ease-of-use          175        108          1         66    0.6171    0.0057    0.3771      0.1938          0.1196
      email           40         33          0          7    0.8250    0.0000    0.1750      0.0443          0.0365
      phone           40         36          0          4    0.9000    0.0000    0.1000      0.0443          0.0399
    reviews           40         34          3          3   

In [40]:
# ANSWER NEEDS: the priority score of every eligible topic
t = start("T8-H1")

t.filter(org="Vella & Co")                              # -> s1
# the floor this question names is on negative reviews, not mentions
t.summarise("s1", group_by="aspect", min_n=MIN_N, min_n_column="neg_count")   # -> t1
t.rank("t1", by="priority_score", top_k=None)       # -> t2, prevalence x neg_rate

report(t)

T8-H1  |  dataset: hard
Which 2 issues should Vella & Co address first? Rank aspects using this formula: priority score =
aspect prevalence x aspect negative rate. Ignore any aspects with less than 30 negative reviews.

[1] filter(org='Vella & Co')
     {'n_rows': 777}

[2] summarise(ref='s1', group_by='aspect', min_n=30, min_n_column='neg_count')
group  total_count  neg_count  neu_count  pos_count  neg_rate  neu_rate  pos_rate  prevalence  priority_score
speed          150         44          0        106    0.2933     0.000    0.7067      0.1931          0.0566
phone           40         32          3          5    0.8000     0.075    0.1250      0.0515          0.0412
     selection_rows: 777
     below_min_n: ['account-access', 'app-website', 'attitude-of-staff', 'competitor', 'discounts-promotions', 'email', 'general-satisfaction', 'price-value-for-money', 'reviews']

[3] rank(ref='t1', by='priority_score', top_k=None)
group  total_count  neg_count  neu_count  pos_count  neg_rate 

In [41]:
# ANSWER NEEDS: the priority score of every eligible topic, and which of them are actionable
t = start("T8-H2")

t.filter(org="CompareHive")                              # -> s1
# the floor this question names is on negative reviews, not mentions
t.summarise("s1", group_by="aspect", min_n=MIN_N, min_n_column="neg_count")   # -> t1
t.rank("t1", by="priority_score", top_k=None)       # -> t2, prevalence x neg_rate

report(t)

T8-H2  |  dataset: hard
Which 2 issues should CompareHive address first? Rank aspects using this formula: priority score =
aspect prevalence x aspect negative rate. Ignore any aspects with less than 30 negative reviews.

[1] filter(org='CompareHive')
     {'n_rows': 779}

[2] summarise(ref='s1', group_by='aspect', min_n=30, min_n_column='neg_count')
               group  total_count  neg_count  neu_count  pos_count  neg_rate  neu_rate  pos_rate  prevalence  priority_score
general-satisfaction          116         31          1         84    0.2672    0.0086    0.7241      0.1489          0.0398
         app-website           80         46          0         34    0.5750    0.0000    0.4250      0.1027          0.0591
     selection_rows: 779
     below_min_n: ['account-access', 'attitude-of-staff', 'competitor', 'discounts-promotions', 'ease-of-use', 'email', 'phone', 'price-value-for-money', 'reviews', 'speed']

[3] rank(ref='t1', by='priority_score', top_k=None)
               group 

In [42]:
# ANSWER NEEDS: the priority score of every eligible topic
t = start("T8-H3")

t.filter(org="Kerbside")                              # -> s1
# the floor this question names is on negative reviews, not mentions
t.summarise("s1", group_by="aspect", min_n=MIN_N, min_n_column="neg_count")   # -> t1
t.rank("t1", by="priority_score", top_k=None)       # -> t2, prevalence x neg_rate

report(t)

T8-H3  |  dataset: hard
Which 2 issues should Kerbside address first? Rank aspects using this formula: priority score =
aspect prevalence x aspect negative rate. Ignore any aspects with less than 30 negative reviews.

[1] filter(org='Kerbside')
     {'n_rows': 248}

[2] summarise(ref='s1', group_by='aspect', min_n=30, min_n_column='neg_count')
               group  total_count  neg_count  neu_count  pos_count  neg_rate  neu_rate  pos_rate  prevalence  priority_score
         app-website           61         32          4         25    0.5246    0.0656    0.4098      0.2460           0.129
general-satisfaction           51         30          1         20    0.5882    0.0196    0.3922      0.2056           0.121
               email           40         31          0          9    0.7750    0.0000    0.2250      0.1613           0.125
     selection_rows: 248
     below_min_n: ['account-access', 'attitude-of-staff', 'competitor', 'discounts-promotions', 'ease-of-use', 'phone', 'price-va

## T9 - Comparative diagnosis and recommendation

Note:
- In v1 of the tool set, the agent is expected to compare the negative rates per aspect of the 2 companies without the use of any tools. If agents fail at this, then we can add another tool for comparison.

In [43]:
# ANSWER NEEDS: A's worst topics, B's rate on each of them, and which of those topics A can act on
t = start("T9-E1")

t.filter(org="Halden Savings")                                # -> s1
t.summarise("s1", group_by="aspect", min_n=MIN_N)   # -> t1
t.rank("t1", by="neg_rate", top_k=3)                # -> t2, A's top 3 topics
t.filter(org="Kestrel Bank")                                # -> s2
t.summarise("s2", group_by="aspect", min_n=MIN_N)   # -> t3, B on the same topics

report(t)

T9-E1  |  dataset: easy
Why does Halden Savings have a higher complaint rate than Kestrel Bank? What should Halden Savings
improve first?

[1] filter(org='Halden Savings')
     {'n_rows': 571}

[2] summarise(ref='s1', group_by='aspect', min_n=30)
                group  total_count  neg_count  neu_count  pos_count  neg_rate  neu_rate  pos_rate  prevalence  priority_score
 general-satisfaction           85         32          0         53    0.3765    0.0000    0.6235      0.1489          0.0560
          ease-of-use           64         14          0         50    0.2188    0.0000    0.7812      0.1121          0.0245
          app-website           62         24         12         26    0.3871    0.1935    0.4194      0.1086          0.0420
       account-access           40         32          7          1    0.8000    0.1750    0.0250      0.0701          0.0560
           competitor           40          8          0         32    0.2000    0.0000    0.8000      0.0701          0.01

In [44]:
# ANSWER NEEDS: A's worst topics, B's rate on each of them, and which of those topics A can act on
t = start("T9-E2")

t.filter(org="CompareHive")                                # -> s1
t.summarise("s1", group_by="aspect", min_n=MIN_N)   # -> t1
t.rank("t1", by="neg_rate", top_k=3)                # -> t2, A's top 3 topics
t.filter(org="PricePilot")                                # -> s2
t.summarise("s2", group_by="aspect", min_n=MIN_N)   # -> t3, B on the same topics

report(t)

T9-E2  |  dataset: easy
Why does CompareHive have a higher complaint rate than PricePilot? What should CompareHive improve
first?

[1] filter(org='CompareHive')
     {'n_rows': 986}

[2] summarise(ref='s1', group_by='aspect', min_n=30)
                group  total_count  neg_count  neu_count  pos_count  neg_rate  neu_rate  pos_rate  prevalence  priority_score
 general-satisfaction          190         58          1        131    0.3053    0.0053    0.6895      0.1927          0.0588
          ease-of-use          147         24          0        123    0.1633    0.0000    0.8367      0.1491          0.0243
price-value-for-money          135         21          0        114    0.1556    0.0000    0.8444      0.1369          0.0213
    attitude-of-staff          103         11          2         90    0.1068    0.0194    0.8738      0.1045          0.0112
          app-website          100         74          0         26    0.7400    0.0000    0.2600      0.1014          0.0751
        

In [45]:
# ANSWER NEEDS: A's worst topics, B's rate on each of them, and which of those topics A can act on
t = start("T9-H1")

t.filter(org="Sable Row")                                # -> s1
t.summarise("s1", group_by="aspect", min_n=MIN_N)   # -> t1
t.rank("t1", by="neg_rate", top_k=3)                # -> t2, A's top 3 topics
t.filter(org="Northerly")                                # -> s2
t.summarise("s2", group_by="aspect", min_n=MIN_N)   # -> t3, B on the same topics

report(t)

T9-H1  |  dataset: hard
Why does Sable Row have a higher complaint rate than Northerly? What should Sable Row improve first?



[1] filter(org='Sable Row')
     {'n_rows': 925}

[2] summarise(ref='s1', group_by='aspect', min_n=30)
                group  total_count  neg_count  neu_count  pos_count  neg_rate  neu_rate  pos_rate  prevalence  priority_score
 general-satisfaction          354        109          0        245    0.3079    0.0000    0.6921      0.3827          0.1178
          app-website          253         46          2        205    0.1818    0.0079    0.8103      0.2735          0.0497
                speed          100         30          0         70    0.3000    0.0000    0.7000      0.1081          0.0324
          ease-of-use           74         21          0         53    0.2838    0.0000    0.7162      0.0800          0.0227
    attitude-of-staff           56         44          0         12    0.7857    0.0000    0.2143      0.0605          0.0476
price-value-for-money           40          8          0         32    0.2000    0.0000    0.8000      0.0432          0.0086
     selection

In [46]:
# ANSWER NEEDS: A's worst topics, B's rate on each of them, and which of those topics A can act on
t = start("T9-H2")

t.filter(org="Larkmead Market")                                # -> s1
t.summarise("s1", group_by="aspect", min_n=MIN_N)   # -> t1
t.rank("t1", by="neg_rate", top_k=3)                # -> t2, A's top 3 topics
t.filter(org="Oakpan Grocers")                                # -> s2
t.summarise("s2", group_by="aspect", min_n=MIN_N)   # -> t3, B on the same topics

report(t)

T9-H2  |  dataset: hard
Why does Larkmead Market have a higher complaint rate than Oakpan Grocers? What should Larkmead
Market improve first?

[1] filter(org='Larkmead Market')
     {'n_rows': 658}

[2] summarise(ref='s1', group_by='aspect', min_n=30)
               group  total_count  neg_count  neu_count  pos_count  neg_rate  neu_rate  pos_rate  prevalence  priority_score
         app-website          189         87          2        100    0.4603    0.0106    0.5291      0.2872          0.1322
         ease-of-use          170         98          1         71    0.5765    0.0059    0.4176      0.2584          0.1489
general-satisfaction           96         33          1         62    0.3438    0.0104    0.6458      0.1459          0.0502
discounts-promotions           54         22          1         31    0.4074    0.0185    0.5741      0.0821          0.0334
               speed           51         29          0         22    0.5686    0.0000    0.4314      0.0775          0.044

In [47]:
# ANSWER NEEDS: A's worst topics, B's rate on each of them, and which of those topics A can act on
t = start("T9-H3")

t.filter(org="Halden Savings")                                # -> s1
t.summarise("s1", group_by="aspect", min_n=MIN_N)   # -> t1
t.rank("t1", by="neg_rate", top_k=3)                # -> t2, A's top 3 topics
t.filter(org="Northeast Bank")                                # -> s2
t.summarise("s2", group_by="aspect", min_n=MIN_N)   # -> t3, B on the same topics

report(t)

T9-H3  |  dataset: hard
Why does Halden Savings have a higher complaint rate than Northeast Bank? What should Halden Savings
improve first?

[1] filter(org='Halden Savings')
     {'n_rows': 336}

[2] summarise(ref='s1', group_by='aspect', min_n=30)
               group  total_count  neg_count  neu_count  pos_count  neg_rate  neu_rate  pos_rate  prevalence  priority_score
         ease-of-use          107         28          0         79    0.2617    0.0000    0.7383      0.3185          0.0833
general-satisfaction           80         31          1         48    0.3875    0.0125    0.6000      0.2381          0.0923
               speed           40          6          0         34    0.1500    0.0000    0.8500      0.1190          0.0179
     selection_rows: 336
     below_min_n: ['account-access', 'attitude-of-staff', 'competitor', 'discounts-promotions', 'email', 'price-value-for-money', 'reviews']

[3] rank(ref='t1', by='neg_rate', top_k=3)
               group  total_count  neg_co

## T10 - Full CX report

In [48]:
# ANSWER NEEDS: the sentiment mix, the top 3 issues, both joint-first issues tested against the industry, and a roadmap
t = start("T10-E1")

# account access and email support are joint first, so both are tested
t.filter(org="Trippa")                                # -> s1
t.summarise("s1")                                     # -> t1, the sentiment mix
t.summarise("s1", group_by="aspect", min_n=MIN_N)      # -> t2
t.rank("t2", by="neg_rate", top_k=3)                  # -> t3, the top issues

# account-access vs the industry. The industry slice contains Trippa's own reviews,
# so the test is run against the rest of the industry; the average including
# Trippa is computed separately because the report quotes it.
t.filter(org="Trippa", aspect="account-access")   # -> s2
t.filter(industry="Travel Booking", aspect="account-access")   # -> s3
t.summarise("s3")                                     # -> t4, Travel Booking average, Trippa included
t.filter(industry="Travel Booking", aspect="account-access", exclude_org="Trippa")   # -> s4
t.ztest("s2", "s4")                              # Trippa vs the rest of Travel Booking

# email vs the industry. The industry slice contains Trippa's own reviews,
# so the test is run against the rest of the industry; the average including
# Trippa is computed separately because the report quotes it.
t.filter(org="Trippa", aspect="email")   # -> s5
t.filter(industry="Travel Booking", aspect="email")   # -> s6
t.summarise("s6")                                     # -> t5, Travel Booking average, Trippa included
t.filter(industry="Travel Booking", aspect="email", exclude_org="Trippa")   # -> s7
t.ztest("s5", "s7")                              # Trippa vs the rest of Travel Booking

t.rank("t3", by="neg_rate", top_k=None, exclude="non_actionable")   # -> t6, roadmap order

report(t)

T10-E1  |  dataset: easy
Write a CX report for Trippa. Cover overall sentiment, the top 3 issues ranked by negative sentiment
rate, and how the top issue compares with the industry average using a statistical significance
test. Also give me an improvement roadmap.

[1] filter(org='Trippa')
     {'n_rows': 674}

[2] summarise(ref='s1')
group  total_count  neg_count  neu_count  pos_count  neg_rate  neu_rate  pos_rate  prevalence  priority_score
  all          674        285         10        379    0.4228    0.0148    0.5623         1.0          0.4228
     selection_rows: 674

[3] summarise(ref='s1', group_by='aspect', min_n=30)
                group  total_count  neg_count  neu_count  pos_count  neg_rate  neu_rate  pos_rate  prevalence  priority_score
          app-website          123         31          2         90    0.2520    0.0163    0.7317      0.1825          0.0460
 general-satisfaction          123         54          0         69    0.4390    0.0000    0.5610      0.1825   

In [49]:
# ANSWER NEEDS: the sentiment mix, the top 3 issues, the top issue tested against the industry, and a roadmap
t = start("T10-E2")

t.filter(org="Kestrel Bank")                                # -> s1
t.summarise("s1")                                     # -> t1, the sentiment mix
t.summarise("s1", group_by="aspect", min_n=MIN_N)      # -> t2
t.rank("t2", by="neg_rate", top_k=3)                  # -> t3, the top issues

# attitude-of-staff vs the industry. The industry slice contains Kestrel Bank's own reviews,
# so the test is run against the rest of the industry; the average including
# Kestrel Bank is computed separately because the report quotes it.
t.filter(org="Kestrel Bank", aspect="attitude-of-staff")   # -> s2
t.filter(industry="Banking", aspect="attitude-of-staff")   # -> s3
t.summarise("s3")                                     # -> t4, Banking average, Kestrel Bank included
t.filter(industry="Banking", aspect="attitude-of-staff", exclude_org="Kestrel Bank")   # -> s4
t.ztest("s2", "s4")                              # Kestrel Bank vs the rest of Banking

t.rank("t3", by="neg_rate", top_k=None, exclude="non_actionable")   # -> t5, roadmap order

report(t)

T10-E2  |  dataset: easy
Write a CX report for Kestrel Bank. Cover overall sentiment, the top 3 issues ranked by negative
sentiment rate, and how the top issue compares with the industry average using a statistical
significance test. Also give me an improvement roadmap.

[1] filter(org='Kestrel Bank')
     {'n_rows': 709}

[2] summarise(ref='s1')
group  total_count  neg_count  neu_count  pos_count  neg_rate  neu_rate  pos_rate  prevalence  priority_score
  all          709        276         68        365    0.3893    0.0959    0.5148         1.0          0.3893
     selection_rows: 709

[3] summarise(ref='s1', group_by='aspect', min_n=30)
                group  total_count  neg_count  neu_count  pos_count  neg_rate  neu_rate  pos_rate  prevalence  priority_score
          app-website          139         17         43         79    0.1223    0.3094    0.5683      0.1961          0.0240
 general-satisfaction          110         31          0         79    0.2818    0.0000    0.7182   

In [50]:
# ANSWER NEEDS: the sentiment mix, the top 3 issues, the top issue tested against the industry, and a roadmap
t = start("T10-H1")

t.filter(org="Vella & Co")                                # -> s1
t.summarise("s1")                                     # -> t1, the sentiment mix
t.summarise("s1", group_by="aspect", min_n=MIN_N)      # -> t2
t.rank("t2", by="neg_rate", top_k=3)                  # -> t3, the top issues

# phone vs the industry. The industry slice contains Vella & Co's own reviews,
# so the test is run against the rest of the industry; the average including
# Vella & Co is computed separately because the report quotes it.
t.filter(org="Vella & Co", aspect="phone")   # -> s2
t.filter(industry="Fashion", aspect="phone")   # -> s3
t.summarise("s3")                                     # -> t4, Fashion average, Vella & Co included
t.filter(industry="Fashion", aspect="phone", exclude_org="Vella & Co")   # -> s4
t.ztest("s2", "s4")                              # Vella & Co vs the rest of Fashion

t.rank("t3", by="neg_rate", top_k=None, exclude="non_actionable")   # -> t5, roadmap order

report(t)

T10-H1  |  dataset: hard
Write a CX report for Vella & Co. Cover overall sentiment, the top 3 issues ranked by negative
sentiment rate, and how the top issue compares with the industry average using a statistical
significance test. Also give me an improvement roadmap.

[1] filter(org='Vella & Co')
     {'n_rows': 777}

[2] summarise(ref='s1')
group  total_count  neg_count  neu_count  pos_count  neg_rate  neu_rate  pos_rate  prevalence  priority_score
  all          777        189          9        579    0.2432    0.0116    0.7452         1.0          0.2432
     selection_rows: 777

[3] summarise(ref='s1', group_by='aspect', min_n=30)
                group  total_count  neg_count  neu_count  pos_count  neg_rate  neu_rate  pos_rate  prevalence  priority_score
 general-satisfaction          247          6          2        239    0.0243    0.0081    0.9676      0.3179          0.0077
          app-website          156         29          1        126    0.1859    0.0064    0.8077      0

In [51]:
# ANSWER NEEDS: the sentiment mix, the top 3 issues, the top issue tested against the industry, and a roadmap
t = start("T10-H2")

t.filter(org="Freshbury")                                # -> s1
t.summarise("s1")                                     # -> t1, the sentiment mix
t.summarise("s1", group_by="aspect", min_n=MIN_N)      # -> t2
t.rank("t2", by="neg_rate", top_k=3)                  # -> t3, the top issues

# app-website vs the industry. The industry slice contains Freshbury's own reviews,
# so the test is run against the rest of the industry; the average including
# Freshbury is computed separately because the report quotes it.
t.filter(org="Freshbury", aspect="app-website")   # -> s2
t.filter(industry="Groceries", aspect="app-website")   # -> s3
t.summarise("s3")                                     # -> t4, Groceries average, Freshbury included
t.filter(industry="Groceries", aspect="app-website", exclude_org="Freshbury")   # -> s4
t.ztest("s2", "s4")                              # Freshbury vs the rest of Groceries

t.rank("t3", by="neg_rate", top_k=None, exclude="non_actionable")   # -> t5, roadmap order

report(t)

T10-H2  |  dataset: hard
Write a CX report for Freshbury. Cover overall sentiment, the top 3 issues ranked by negative
sentiment rate, and how the top issue compares with the industry average using a statistical
significance test. Also give me an improvement roadmap.

[1] filter(org='Freshbury')
     {'n_rows': 677}

[2] summarise(ref='s1')
group  total_count  neg_count  neu_count  pos_count  neg_rate  neu_rate  pos_rate  prevalence  priority_score
  all          677        263         13        401    0.3885    0.0192    0.5923         1.0          0.3885
     selection_rows: 677

[3] summarise(ref='s1', group_by='aspect', min_n=30)
               group  total_count  neg_count  neu_count  pos_count  neg_rate  neu_rate  pos_rate  prevalence  priority_score
         app-website          262        130          5        127    0.4962    0.0191    0.4847      0.3870          0.1920
         ease-of-use          189         57          0        132    0.3016    0.0000    0.6984      0.2792

In [52]:
# ANSWER NEEDS: the sentiment mix and the industry comparison, which are computable, plus proof that only one topic clears the floor so there is no top 3 and no roadmap
t = start("T10-H3")

# both summarise calls are kept: the unfiltered one shows how few topics
# Lumora is mentioned on at all, the floored one shows how few are reportable
t.filter(org="Lumora")                                # -> s1
t.summarise("s1")                                     # -> t1, the sentiment mix
t.summarise("s1", group_by="aspect")                  # -> t2, every topic the org is mentioned on
t.summarise("s1", group_by="aspect", min_n=MIN_N)      # -> t3
t.rank("t3", by="neg_rate", top_k=3)                  # -> t4, the top issues

# account-access vs the industry. The industry slice contains Lumora's own reviews,
# so the test is run against the rest of the industry; the average including
# Lumora is computed separately because the report quotes it.
t.filter(org="Lumora", aspect="account-access")   # -> s2
t.filter(industry="Streaming", aspect="account-access")   # -> s3
t.summarise("s3")                                     # -> t5, Streaming average, Lumora included
t.filter(industry="Streaming", aspect="account-access", exclude_org="Lumora")   # -> s4
t.ztest("s2", "s4")                              # Lumora vs the rest of Streaming

report(t)

T10-H3  |  dataset: hard
Write a CX report for Lumora. Cover overall sentiment, the top 3 issues ranked by negative sentiment
rate, and how the top issue compares with the industry average using a statistical significance
test. Also give me an improvement roadmap.

[1] filter(org='Lumora')
     {'n_rows': 56}

[2] summarise(ref='s1')
group  total_count  neg_count  neu_count  pos_count  neg_rate  neu_rate  pos_rate  prevalence  priority_score
  all           56         18         32          6    0.3214    0.5714    0.1071         1.0          0.3214
     selection_rows: 56

[3] summarise(ref='s1', group_by='aspect')
               group  total_count  neg_count  neu_count  pos_count  neg_rate  neu_rate  pos_rate  prevalence  priority_score
      account-access           40         14         25          1    0.3500     0.625    0.0250      0.7143          0.2500
         app-website           12          4          6          2    0.3333     0.500    0.1667      0.2143          0.0714
d

## Compile gold tool paths


In [53]:
summary = [{"task_id": task_id,
            "template": task_id.split("-")[0],
            "dataset": questions.loc[questions.task_id == task_id, "dataset"].iloc[0],
            "question": questions.loc[questions.task_id == task_id, "question"].iloc[0],
            "gold_answer": GOLD_ANSWERS[task_id],
            "gold_tool_path": [signature(call) for call in tools.calls],
            "gold_num_steps": len(tools.calls)}   for task_id, tools in RUNS.items()]

gold_tool_paths = pd.DataFrame(summary)
gold_tool_paths

,task_id,template,dataset,question,gold_answer,gold_tool_path,gold_num_steps
0,T1-E1,T1,easy,What proportion of Banking reviews that mentio...,18.7% of Banking reviews about the app or webs...,"[filter(industry='Banking', aspect='app-websit...",2
1,T1-E2,T1,easy,What proportion of Fashion reviews that mentio...,86.8% of Fashion reviews about ease of use are...,"[filter(industry='Fashion', aspect='ease-of-us...",2
2,T1-H1,T1,hard,What proportion of Fashion reviews that mentio...,21.3% of Fashion reviews about speed are negat...,"[filter(industry='Fashion', aspect='speed'), s...",2
3,T1-H2,T1,hard,What proportion of Price Comparison reviews th...,10.9% of Price Comparison reviews about staff ...,"[filter(industry='Price Comparison', aspect='a...",2
4,T1-H3,T1,hard,What proportion of Ride Hailing reviews that m...,No answer. Ride Hailing has only 3 reviews men...,"[filter(industry='Ride Hailing', aspect='disco...",2
5,T2-E1,T2,easy,What is the sentiment breakdown across all Rid...,The customer base is mostly dissatisfied. Ride...,"[filter(industry='Ride Hailing'), summarise(re...",2
6,T2-E2,T2,easy,What is the sentiment breakdown across all Mar...,The customer base is mostly satisfied. Marbroo...,"[filter(org='Marbrook'), summarise(ref='s1')]",2
7,T2-H1,T2,hard,What is the sentiment breakdown across all Van...,The customer base is mostly satisfied. Vanter ...,"[filter(org='Vanter Financial'), summarise(ref...",2
8,T2-H2,T2,hard,What is the sentiment breakdown across all Pri...,The customer base is mostly satisfied. PricePi...,"[filter(org='PricePilot'), summarise(ref='s1')]",2
9,T2-H3,T2,hard,What is the sentiment breakdown across all Sab...,The customer base is mostly satisfied. Sable R...,"[filter(org='Sable Row'), summarise(ref='s1')]",2


In [54]:
gold_tool_paths.to_csv('../benchmark_outputs/questions_and_answers.csv')